In [ ]:
import pennylane as qml
import numpy as np

# ==========================================
# PARTE 1: I DATI E I REGISTRI/FILI
# ==========================================
# 1. Partiamo da una VERA matrice diagonale 4x4 (L'input)
# X_matrice = np.array([
#     [ 0.5,   0,   0,   0],
#     [  0, -0.2,   0,   0],
#     [  0,   0, 0.8,   0],
#     [  0,   0,   0,   0.1]
# ])

X_matrice = np.array([
    [ 0.7,   0,   0,   0],
    [  0, -4,   0,   0],
    [  0,   0, 8,   0],
    [  0,   0,   0,   0.3]
])

# X_matrice = np.array([
#     [ -720,   0,   0,   0],
#     [  0, 56,   0,   0],
#     [  0,   0, 200,   0],
#     [  0,   0,   0,   170]
# ])

# X_matrice = np.array([
#     [ -720,   0],
#     [  0, 56]
# ])

# X_matrice = np.array([
#     [ -720,   0,   0],
#     [  0, 56,   0],
#     [  0,   0, 200],
# ])

print("1. La matrice X di partenza è:")
print(X_matrice)


# 2. Estraiamo la diagonale classica
x_classico_grezzo = np.diag(X_matrice)
N_originale = len(x_classico_grezzo)


# 3. Calcolo qubit e Padding
# Calcoliamo n: il numero di qubit necessari (logaritmo in base 2 arrotondato per eccesso)
# Es: N=2 -> n=1. N=3 -> n=2. N=4 -> n=2.
n = int(np.ceil(np.log2(N_originale)))

# Calcoliamo quanti slot fisici avrà la memoria quantistica (la potenza di 2 più vicina)
N_pad = 2**n

# Facciamo il padding: aggiungiamo zeri alla fine se N_originale è minore di N_pad
# (Fondamentale per le matrici 3x3, 5x5, ecc.)
x_classico = np.pad(x_classico_grezzo, (0, N_pad - N_originale))


# 4. Normalizzazione del vettore (obbligatoria per le ampiezze, sennò il computer quantistico non può leggere i dati)
norma_originale = np.linalg.norm(x_classico)
x = x_classico / norma_originale 


# 5. Assegniamo i fili. n=2 (perché abbiamo 4 dati, 2^2=4)
# Servono n fili per address, n per data, 1 per l'ancilla B, 1 per l'ancilla lcu.
# Generiamo le liste dei fili in base alla 'n' calcolata!
ancilla_lcu = [0]                                    # Il nuovo qubit (+1) in alto
qubits_address = list(range(1, n+1))                   # Es: se n=1 -> [0]
qubits_dati = list(range(n+1, 2*n +1))                    # Es: se n=1 -> [1]
ancilla_b = [2*n +1]                                    # Es: se n=1 -> [2]

# Prepariamo un raggruppamento che ci servirà per S_0
qubits_da_B = qubits_dati + ancilla_b

# Creiamo una lista unica con tutti i 5 fili nell'ordine corretto
tutti_i_fili = ancilla_lcu + qubits_address + qubits_dati + ancilla_b



# ==========================================
# PARTE 2: ORACOLO E RIFLESSIONE
# ==========================================

def U_x(wires):
    """
    L'Oracolo: Inietta il vettore x nelle ampiezze dei qubit bersaglio.
    """
     #qml.AmplitudeEmbedding(features=x, wires=wires, normalize=True)
    qml.MottonenStatePreparation(x, wires=wires)

def S_0():
    """
    La Riflessione: La matrice diagonale con un -1 in alto a sinistra.
    Inverte il segno SOLO se i qubit 'dati' e 'B' sono tutti a zero.
    Non tocca gli 'address' per non rovinare le coordinate!
    """
    stato_zero = np.zeros(len(qubits_da_B), dtype=int)
    qml.FlipSign(stato_zero, wires=qubits_da_B)



# ==========================================
# PARTE 3: W (dalla Figura 1)
# ==========================================

def W():
    """
    Allinea i dati x_k con le coordinate corrette della matrice diagonale.
    """
    # 1. U normale solo sui dati
    U_x(wires=qubits_dati)
    
    # 2. Hadamard sull'ancilla B
    qml.Hadamard(wires=ancilla_b)
    
    # 3. U inverso sui dati, controllato da B
    # qml.adjoint ribalta l'oracolo, qml.ctrl lo subordina a B
    qml.ctrl(qml.adjoint(U_x), control=ancilla_b)(wires=qubits_dati)
    
    # 4. Le porte Toffoli incrociate (da address a dati, controllate da B)
    # Servono a "saldare" il dato corretto all'indirizzo corretto
    for i in range(len(qubits_address)):
        # Toffoli prende: [controllo_1, controllo_2, bersaglio]
        qml.Toffoli(wires=[ancilla_b[0], qubits_address[i], qubits_dati[i]])
        
    # 5. Hadamard finale sull'ancilla B
    qml.Hadamard(wires=ancilla_b)



# ==========================================
# PARTE 4: G
# ==========================================

def G():
    # Costruisce l'operatore G = W S_0 W^\dagger Z_B.
    # Le operazioni si scrivono nell'ordine esatto in cui toccano i qubit.
    
    # 1. Z_B (Riflessione sull'ancilla)
    qml.PauliZ(wires=ancilla_b)
    
    # 2. W^\dagger (Smontaggio: PennyLane inverte tutto W in automatico)
    qml.adjoint(W)()
    
    # 3. S_0 (Riflessione sull'origine)
    S_0()
    
    # 4. W (Rimontaggio)
    W()



# ==========================================
# PARTE 5: Gtilde
# ==========================================

def G_tilde():
    """Implementa -1/2 (G + G_dagger) usando le porte esatte del paper"""
    
    # 1. Hadamard iniziale sull'ancilla LCU
    qml.Hadamard(wires=ancilla_lcu)
    
    # 2. Sandwich PauliX per il controllo su 0 (Pallino bianco)
    qml.PauliX(wires=ancilla_lcu)
    qml.ctrl(G, control=ancilla_lcu)()  # G controllato standard
    qml.PauliX(wires=ancilla_lcu)
    
    # 3. Controllo su 1 per G_dagger (Pallino nero)
    qml.ctrl(qml.adjoint(G), control=ancilla_lcu)()
    
    # 4. Hadamard finale per ricongiungere gli stati
    qml.Hadamard(wires=ancilla_lcu)
    
    # 5. La rotazione Rz(2pi) per ottenere il segno MENO globale
    qml.RZ(2 * np.pi, wires=ancilla_lcu)
